# linalg-solve-batched — worked example 1: Solve a batch of 3×3 systems and verify by back-substitution

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `linalg-solve-batched`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`torch.linalg.solve(A, b)` accepts arbitrary leading batch dimensions and solves all systems simultaneously. For `A: (K, n, n)` and `b: (K, n)`, it returns `x: (K, n)` where each `x[k]` satisfies `A[k] @ x[k] = b[k]`. The result can be verified by computing `A @ x` and checking against `b`.

## Worked solution

Step 1: Build a batch of 5 non-singular 3×3 matrices by starting from random matrices and adding a scaled identity to ensure invertibility.

Step 2: Generate right-hand-side vectors `b` of shape `(5, 3)`.

Step 3: Call `x = t.linalg.solve(A, b)`. The output has shape `(5, 3)`.

Step 4: Verify by computing `A @ x.unsqueeze(-1)` (or using `torch.einsum`) and checking that the result matches `b` to floating-point tolerance.

In [ ]:
import torch as t

t.manual_seed(0)

K, n = 5, 3

# Build non-singular batch: random + scaled identity
A = t.randn(K, n, n)
A = A + 2 * t.eye(n).unsqueeze(0)   # ensures invertibility
b = t.randn(K, n)

# Solve all K systems at once
x = t.linalg.solve(A, b)
print(f'x shape: {x.shape}')   # (5, 3)

# Verify: A @ x should equal b
Ax = (A @ x.unsqueeze(-1)).squeeze(-1)   # (K, n)
residual = (Ax - b).abs().max().item()
print(f'max residual: {residual:.2e}')   # ~1e-6 or smaller
assert residual < 1e-4, f'residual too large: {residual}'
print('Verification passed.')